# COURSE PROJECT: PROGRAMMING FOUNDATIONS FOR DATA ANALYSIS AND VISUALIZATION
**Semester 261 – Academic Year 2026–2027** | **Instructor:** Lê Thành Sách

---
## PART 3: TIME SERIES DATA
### Topic: 24-Hour-Ahead PM2.5 Forecasting — Hanoi vs Ho Chi Minh City

- **Group:** [Group name]
- **Member in charge:** [Full name – Student ID]

**Problem.** At time $t$, using the PM2.5 history and the current weather, forecast the PM2.5 concentration
at time $t + 24h$ (the same hour tomorrow). This is a **time series regression** problem.

**Why compare two cities?** Hanoi (four seasons, winter temperature inversion) and Ho Chi Minh City (HCMC; wet/dry seasons)
have different pollution mechanisms. We apply **the same pipeline** to each city **independently** (no merging) to answer:
*how much does machine learning improve over a naive baseline in each city, and which city is harder to forecast?*

## 0. ENVIRONMENT SETUP

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

RANDOM_STATE = 42
HORIZON = 24            # forecast 24 hours ahead
WHO_24H = 15            # WHO 2021 guideline for 24-hour mean PM2.5 (µg/m³)
CITY_COLORS = {'Hanoi': '#d62728', 'HCMC': '#1f77b4'}
import sklearn
print(f'Environment ready! (scikit-learn {sklearn.__version__}, pandas {pd.__version__})')

## 1. DATASET DESCRIPTION

### 1.1. Data sources (self-collected)
The data was **collected through public APIs** with the script `scripts/crawl_air_quality.py`:

| Component | Source | Notes |
|---|---|---|
| PM2.5 – Hanoi | [OpenAQ v3](https://openaq.org) – station *46 Luu Quang Vu* (Hanoi Air Quality Monitoring Network, sensor 7772032) | City monitoring station, **has real gaps** |
| PM2.5 – HCMC | OpenAQ v3 – station *CMT8* (AirGradient, sensor 11357424) | Low-cost sensor |
| Weather | [Open-Meteo Historical Weather API](https://open-meteo.com) | Temperature, humidity, precipitation, wind, pressure, cloud cover |

- **Period:** 01/12/2024 – 30/09/2026 (22 months, 16,056 hours per city), Vietnam local time (UTC+7).
- A complete hourly time grid is built: hours with no measurement become `NaN`.

| Column | Type | Meaning |
|---|---|---|
| `datetime` | datetime | Timestamp (local time) |
| `city` | categorical | City |
| `temperature_2m` | numerical | Temperature at 2 m (°C) |
| `relative_humidity_2m` | numerical | Relative humidity (%) |
| `precipitation` | numerical | Precipitation (mm) |
| `wind_speed_10m` | numerical | Wind speed at 10 m (km/h) |
| `wind_direction_10m` | numerical (angle) | Wind direction (degrees) → converted to a **categorical** 8-sector feature |
| `surface_pressure` | numerical | Surface pressure (hPa) |
| `cloud_cover` | numerical | Cloud cover (%) |
| `pm25` | numerical | PM2.5 concentration (µg/m³) — **the variable to forecast** |
| `pm25_source` | categorical | PM2.5 source (`openaq`, or `open-meteo` if the crawler had to fall back) |

### 1.2. Loading the data
The CSV files are read locally; on Colab, if they are not found, they are downloaded from GitHub (**set `GITHUB_RAW_BASE`** to the group's repo).

In [ ]:
GITHUB_RAW_BASE = 'https://raw.githubusercontent.com/perckgg/NTLT/main/03_TimeSeries/data/timeseries'
CITY_FILES = {'Hanoi': 'air_quality_hanoi.csv', 'HCMC': 'air_quality_hcmc.csv'}
LOCAL_DIRS = ['data/timeseries', '../data/timeseries', '/content/data/timeseries', '/content']


def load_city_data(filename):
    for d in LOCAL_DIRS:
        path = os.path.join(d, filename)
        if os.path.exists(path):
            return pd.read_csv(path, parse_dates=['datetime'])
    return pd.read_csv(f'{GITHUB_RAW_BASE}/{filename}', parse_dates=['datetime'])


raw = {city: load_city_data(f).set_index('datetime').sort_index().assign(city=city)
       for city, f in CITY_FILES.items()}
for city, df in raw.items():
    print(f'{city}: {df.shape[0]:,} rows × {df.shape[1]} columns | '
          f'{df.index.min():%Y-%m-%d} → {df.index.max():%Y-%m-%d} | PM2.5 source: {", ".join(df.pm25_source.unique())}')
raw['Hanoi'].head()

### 1.3. Descriptive statistics

In [ ]:
raw['Hanoi'].info()
summary = pd.concat({city: df['pm25'].describe() for city, df in raw.items()}, axis=1).T
summary['missing_rate'] = [df['pm25'].isna().mean() for df in raw.values()]
summary['share of hours > WHO'] = [(df['pm25'] > WHO_24H).mean() for df in raw.values()]
display(summary.round(3))
display(pd.concat({city: df.describe().T for city, df in raw.items()}).round(2))

### 1.4. Time series visualization
**Figure 1 – Daily mean PM2.5** in both cities against the WHO 24-hour guideline (15 µg/m³).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for city, df in raw.items():
    daily = df['pm25'].resample('D').mean()
    ax.plot(daily.index, daily, label=city, color=CITY_COLORS[city], lw=0.9)
ax.axhline(WHO_24H, color='k', ls='--', lw=1, label='WHO 24h guideline (15 µg/m³)')
ax.set(title='Daily mean PM2.5', ylabel='PM2.5 (µg/m³)')
ax.legend()
plt.show()

**Figure 2 – Distribution of hourly PM2.5** and **Figure 3 – Monthly boxplots** (seasonality).

In [ ]:
plot_df = pd.concat([df[['pm25']].assign(city=city) for city, df in raw.items()])
plot_df['month'] = plot_df.index.month
plot_df['hour'] = plot_df.index.hour
plot_df = plot_df.reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 4))
sns.histplot(data=plot_df, x='pm25', hue='city', bins=100, element='step', stat='density',
             common_norm=False, palette=CITY_COLORS, ax=ax)
ax.set(title='Distribution of hourly PM2.5', xlabel='PM2.5 (µg/m³)', xlim=(0, 250))
plt.show()

fig, ax = plt.subplots(figsize=(14, 5))
sns.boxplot(data=plot_df, x='month', y='pm25', hue='city', palette=CITY_COLORS, showfliers=False, ax=ax)
ax.set(title='PM2.5 by month (outliers hidden for readability)', xlabel='Month', ylabel='PM2.5 (µg/m³)')
plt.show()

**Figure 4 – Daily cycle:** mean PM2.5 by hour of day, split into the dry/cool half-year (Nov–Apr) and the wet/hot half-year (May–Oct).

In [ ]:
plot_df['half_year'] = np.where(plot_df['month'].isin([11, 12, 1, 2, 3, 4]), 'Nov–Apr', 'May–Oct')
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, city in zip(axes, CITY_FILES):
    g = plot_df[plot_df.city == city]
    sns.lineplot(data=g, x='hour', y='pm25', hue='half_year', errorbar=None, ax=ax)
    ax.set(title=city, xlabel='Hour of day', ylabel='Mean PM2.5 (µg/m³)', xticks=range(0, 24, 3))
plt.tight_layout()
plt.show()

**Figure 5 – Spearman correlation matrix** between PM2.5 and the weather variables, per city.

In [ ]:
num_cols = ['pm25', 'temperature_2m', 'relative_humidity_2m', 'precipitation',
            'wind_speed_10m', 'surface_pressure', 'cloud_cover']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (city, df) in zip(axes, raw.items()):
    sns.heatmap(df[num_cols].corr(method='spearman'), annot=True, fmt='.2f', cmap='RdBu_r',
                vmin=-1, vmax=1, ax=ax, cbar=False)
    ax.set_title(f'Spearman correlation – {city}')
plt.tight_layout()
plt.show()

**EDA findings:**


- **Similar average pollution, different shapes:** mean PM2.5 is 31.7 µg/m³ in Hanoi and 32.1 µg/m³ in HCMC.
  Hanoi is more volatile (std 20.6 vs 16.3; max 332 vs 171). Both exceed the WHO 24h guideline (15 µg/m³)
  in most hours: 56% in Hanoi, 82% in HCMC.
- **Seasonality:** Hanoi has a clear winter pollution season (December: 53, January: 55 µg/m³) and is cleanest in August (19 µg/m³),
  a difference of almost 3×. HCMC is also most polluted at the end of the dry season / start of the year (January: 46 µg/m³) and cleanest in the middle of the wet season (July: 24 µg/m³), but its range is narrower.
- **Daily cycle:** HCMC peaks in the morning at 7h (≈41 µg/m³), matching the traffic rush hour; Hanoi peaks in the evening (≈34 µg/m³ at 22h),
  consistent with the night-time temperature inversion trapping dust near the ground.
- **Correlation with weather:** in Hanoi PM2.5 correlates negatively with temperature (-0.31) and cloud cover (-0.25) and positively with pressure (+0.31),
  i.e. cold, high-pressure winter air lets dust accumulate. In HCMC the strongest driver is wind speed (-0.36): strong wind disperses dust.
- **Autocorrelation:** the lag-1h autocorrelation is very high (0.92 / 0.88), but at lag 24h it drops to 0.56 / 0.45. Forecasting 24h ahead
  is therefore much harder than forecasting the next hour, and needs weather, season and daily-cycle features.
- **Data quality:** Hanoi is missing 32.8% of PM2.5 hours, concentrated from late 2025 (December 2025 is 97% missing); HCMC is missing 8.4%.
  This must be handled carefully in Section 2.

## 2. DATA PREPARATION

The steps below are written as **reusable functions** and applied **identically** to each city,
but every city is processed **independently**.

### 2.1. Missing value statistics

In [ ]:
def gap_lengths(series):
    """Length (hours) of each run of consecutive missing values."""
    is_na = series.isna()
    groups = (is_na != is_na.shift()).cumsum()
    return is_na.groupby(groups).sum()[lambda s: s > 0]


def missing_report(raw_dict):
    rows = []
    for city, df in raw_dict.items():
        gaps = gap_lengths(df['pm25'])
        rows.append({'City': city, 'Missing hours': int(df['pm25'].isna().sum()),
                     'Missing rate': f"{df['pm25'].isna().mean():.1%}",
                     'Gaps': len(gaps), 'Gaps ≤ 6h': int((gaps <= 6).sum()),
                     'Longest gap (hours)': int(gaps.max()) if len(gaps) else 0})
    return pd.DataFrame(rows)


display(missing_report(raw))
display((pd.concat({city: df.isna().mean() for city, df in raw.items()}, axis=1) * 100)
        .round(2).rename(columns=lambda c: f'{c} (% missing)'))

fig, ax = plt.subplots(figsize=(14, 4))
for city, df in raw.items():
    avail = df['pm25'].notna().resample('MS').mean() * 100
    ax.plot(avail.index, avail, marker='o', label=city, color=CITY_COLORS[city])
ax.set(title='Share of hours with PM2.5 data, by month', ylabel='% of hours with data', ylim=(0, 105))
ax.legend()
plt.show()

### 2.2. Cleaning invalid values and handling missing data

- **Invalid values:** negative PM2.5 or > 1000 µg/m³ are sensor errors → set to `NaN`.
- **"Stuck" sensor:** the same value repeated for ≥ 6 consecutive hours (with PM2.5 > 0) indicates a frozen device → set to `NaN`.
- **Short gaps (≤ 6 hours):** time-based linear interpolation (`interpolate(method='time', limit=6)`) —
  reasonable because PM2.5 changes smoothly over a few hours.
- **Long gaps (> 6 hours):** **not** interpolated (that would fabricate data) → samples whose *target* is missing are dropped;
  remaining missing features are handled by `SimpleImputer` (median, fitted on the training set) inside the pipeline.

In [ ]:
MAX_INTERP_GAP = 6
STUCK_HOURS = 6


def clean_city(df):
    df = df.copy()
    pm = df['pm25']
    invalid = (pm < 0) | (pm > 1000)
    run_id = (pm != pm.shift()).cumsum()
    run_len = pm.groupby(run_id).transform('size')
    stuck = (run_len >= STUCK_HOURS) & (pm > 0) & pm.notna()
    df.loc[invalid | stuck, 'pm25'] = np.nan
    df['pm25_raw'] = df['pm25']
    df['pm25'] = df['pm25'].interpolate(method='time', limit=MAX_INTERP_GAP, limit_area='inside')
    return df, int(invalid.sum()), int(stuck.sum())


clean = {}
for city, df in raw.items():
    clean[city], n_invalid, n_stuck = clean_city(df)
    before, after = df['pm25'].isna().mean(), clean[city]['pm25'].isna().mean()
    print(f'{city}: {n_invalid} negative/>1000 values, {n_stuck} stuck-sensor hours | '
          f'missing PM2.5: {before:.1%} → {after:.1%} after cleaning + interpolation')


def first_short_gap(series, min_len=3, max_len=MAX_INTERP_GAP):
    is_na = series.isna()
    groups = (is_na != is_na.shift()).cumsum()
    sizes = is_na.groupby(groups).transform('sum')
    mask = is_na & (sizes >= min_len) & (sizes <= max_len)
    return series.index[mask][0] if mask.any() else None


for city in clean:
    gap_time = first_short_gap(clean[city]['pm25_raw'])
    if gap_time is not None:
        break
if gap_time is not None:
    window = clean[city].loc[gap_time - pd.Timedelta('36h'): gap_time + pd.Timedelta('36h')]
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(window.index, window['pm25'], 'o--', color='orange', label='After interpolation')
    ax.plot(window.index, window['pm25_raw'], 'o-', color=CITY_COLORS[city], label='Original data')
    ax.set(title=f'Short-gap interpolation example – {city}', ylabel='PM2.5 (µg/m³)')
    ax.legend()
    plt.show()

### 2.3. Outlier detection (IQR)
IQR rule: values outside $[Q1 - 3\cdot IQR,\ Q3 + 3\cdot IQR]$ are treated as extreme outliers.

In [ ]:
def iqr_bounds(s, k=3.0):
    q1, q3 = s.quantile([0.25, 0.75])
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)


fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (city, df) in zip(axes, clean.items()):
    lo, hi = iqr_bounds(df['pm25'])
    out = df.loc[df['pm25'] > hi, 'pm25']
    top_days = out.resample('D').count().nlargest(3).index.strftime('%Y-%m-%d').tolist()
    print(f'{city}: upper bound = {hi:.1f} µg/m³ | {len(out)} outlier hours '
          f'({len(out) / df.pm25.notna().sum():.2%}) | days with most outliers: {top_days}')
    ax.plot(df.index, df['pm25'], lw=0.4, color=CITY_COLORS[city])
    ax.scatter(out.index, out, s=6, color='k', label='Outlier (IQR, k=3)')
    ax.axhline(hi, color='k', ls='--', lw=1)
    ax.set(title=city, ylabel='PM2.5 (µg/m³)')
    ax.legend()
plt.tight_layout()
plt.show()

**Outlier decision:** PM2.5 peaks are **real events** (winter pollution episodes, straw burning, fireworks...),
so they are **not removed** from the target (removing them would make the evaluation look better than it is). Instead, the PM2.5
**features** (lags, rolling statistics) go through a custom `IQRClipper` transformer (bounds learned **on the training set only**)
to reduce the influence of extreme values on the linear model.

### 2.4. Feature engineering

Each sample at time $t$ has target `y = pm25(t + 24h)`. Only information **available at $t$** is used (no future leakage):

| Group | Features | Type |
|---|---|---|
| PM2.5 history | `pm25_lag_0, 1, 2, 3, 6, 12, 23` (value at $t-k$), `pm25_roll_mean_6/24`, `pm25_roll_std_24`, `pm25_diff_24` | numerical |
| Weather at $t$ | temperature, humidity, precipitation, wind speed, pressure, cloud cover, `rain_24h` (total rain over the last 24h) | numerical |
| Calendar of the forecast time | `hour_sin/cos` (cyclic hour encoding) | numerical |
| | `season` (climate season of each city), `day_of_week`, `wind_sector` (8 wind directions) | **categorical** |

> `pm25_lag_0` is the value at the **same hour on the previous day** relative to the forecast time, so the
> **Persistence** baseline (`ŷ(t+24) = pm25(t)`) is also the 24h *seasonal naive* baseline.

In [ ]:
SEASONS = {
    'Hanoi': {12: 'Winter', 1: 'Winter', 2: 'Winter', 3: 'Spring', 4: 'Spring', 5: 'Spring',
              6: 'Summer', 7: 'Summer', 8: 'Summer', 9: 'Autumn', 10: 'Autumn', 11: 'Autumn'},
    'HCMC': {m: ('Dry season' if m in (12, 1, 2, 3, 4) else 'Wet season') for m in range(1, 13)},
}
WIND_SECTORS = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
LAGS = [0, 1, 2, 3, 6, 12, 23]
WEATHER_RAW = ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m',
               'surface_pressure', 'cloud_cover']
WEATHER_FEATURES = WEATHER_RAW + ['rain_24h']
PM_FEATURES = [f'pm25_lag_{k}' for k in LAGS] + ['pm25_roll_mean_6', 'pm25_roll_mean_24',
                                                 'pm25_roll_std_24', 'pm25_diff_24']
CYCLIC_FEATURES = ['hour_sin', 'hour_cos']
CAT_FEATURES = ['season', 'day_of_week', 'wind_sector']
FEATURES = PM_FEATURES + WEATHER_FEATURES + CYCLIC_FEATURES + CAT_FEATURES
TARGET = 'target'


def build_features(df, city, horizon=HORIZON):
    """Build the features at each time t (using only data ≤ t)."""
    pm = df['pm25']
    X = pd.DataFrame(index=df.index)
    for k in LAGS:
        X[f'pm25_lag_{k}'] = pm.shift(k)
    X['pm25_roll_mean_6'] = pm.rolling(6, min_periods=3).mean()
    X['pm25_roll_mean_24'] = pm.rolling(24, min_periods=12).mean()
    X['pm25_roll_std_24'] = pm.rolling(24, min_periods=12).std()
    X['pm25_diff_24'] = pm - pm.shift(24)
    X[WEATHER_RAW] = df[WEATHER_RAW]
    X['rain_24h'] = df['precipitation'].rolling(24, min_periods=1).sum()

    target_time = df.index + pd.Timedelta(hours=horizon)
    X['hour_sin'] = np.sin(2 * np.pi * target_time.hour / 24)
    X['hour_cos'] = np.cos(2 * np.pi * target_time.hour / 24)
    X['season'] = pd.Series(target_time.month, index=df.index).map(SEASONS[city])
    X['day_of_week'] = target_time.day_name()
    sector_idx = ((df['wind_direction_10m'] + 22.5) % 360) // 45
    X['wind_sector'] = sector_idx.map(lambda i: WIND_SECTORS[int(i)] if pd.notna(i) else np.nan)
    return X


def make_dataset(df, city, horizon=HORIZON):
    """Add target = pm25(t + horizon); keep only samples with a target and a current value."""
    X = build_features(df, city, horizon)
    X[TARGET] = df['pm25'].shift(-horizon)
    return X.dropna(subset=[TARGET, 'pm25_lag_0'])


datasets = {city: make_dataset(df, city) for city, df in clean.items()}
for city, d in datasets.items():
    print(f'{city}: {len(d):,} samples | {len(FEATURES)} features '
          f'({len(FEATURES) - len(CAT_FEATURES)} numerical + {len(CAT_FEATURES)} categorical)')
datasets['Hanoi'].head()

### 2.5. Chronological Train / Validation / Test split
A time series **must not be shuffled** (that would leak the future into the past). The split follows **time order**:
first 70% → Train, next 15% → Validation (model selection), last 15% → Test (final evaluation).
An extra `HORIZON` = 24 hours is dropped before each boundary so that the **targets** of one set do not fall inside the next set's period.

In [ ]:
def time_split(d, train_frac=0.70, val_frac=0.15, gap_hours=HORIZON):
    n = len(d)
    t1, t2 = d.index[int(n * train_frac)], d.index[int(n * (train_frac + val_frac))]
    gap = pd.Timedelta(hours=gap_hours)
    train = d[d.index < t1 - gap]
    val = d[(d.index >= t1) & (d.index < t2 - gap)]
    test = d[d.index >= t2]
    return train, val, test


splits = {city: time_split(d) for city, d in datasets.items()}

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, (city, (tr, va, te)) in zip(axes, splits.items()):
    for part, name, color in [(tr, 'Train', 'tab:blue'), (va, 'Validation', 'tab:orange'), (te, 'Test', 'tab:green')]:
        daily = part[TARGET].resample('D').mean()
        ax.plot(daily.index, daily, color=color, lw=0.8,
                label=f'{name}: {len(part):,} samples ({part.index.min():%d/%m/%Y} – {part.index.max():%d/%m/%Y})')
    ax.set(title=city, ylabel='PM2.5 (µg/m³)')
    ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

### 2.6. Preprocessing pipeline (imputation, outlier clipping, encoding, scaling)

| Column group | Steps |
|---|---|
| PM2.5 lags/rolling | `SimpleImputer(median)` → `IQRClipper(k=3)` → `StandardScaler` |
| Weather + hour cycle | `SimpleImputer(median)` → `StandardScaler` |
| Categorical | `SimpleImputer(most_frequent)` → `OneHotEncoder` |

Everything lives inside a `ColumnTransformer` + `Pipeline`, so all parameters (medians, IQR bounds, mean/std, categories)
are **learned on the training set only** → no data leakage.

In [ ]:
class IQRClipper(BaseEstimator, TransformerMixin):
    """Clip (winsorize) each column to [Q1 - k·IQR, Q3 + k·IQR], bounds learned on the fitted data."""

    def __init__(self, k=3.0):
        self.k = k

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        q1, q3 = np.nanpercentile(X, [25, 75], axis=0)
        self.lower_, self.upper_ = q1 - self.k * (q3 - q1), q3 + self.k * (q3 - q1)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)


def build_preprocessor():
    return ColumnTransformer([
        ('pm', Pipeline([('impute', SimpleImputer(strategy='median')),
                         ('clip', IQRClipper(k=3.0)),
                         ('scale', StandardScaler())]), PM_FEATURES),
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                          ('scale', StandardScaler())]), WEATHER_FEATURES + CYCLIC_FEATURES),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                          ('onehot', OneHotEncoder(handle_unknown='ignore'))]), CAT_FEATURES),
    ], sparse_threshold=0)


tr = splits['Hanoi'][0]
pre = build_preprocessor().fit(tr[FEATURES])
Xt = pd.DataFrame(pre.transform(tr[FEATURES]), columns=pre.get_feature_names_out(), index=tr.index)
print(f'After preprocessing: {Xt.shape[1]} columns (from {len(FEATURES)} original columns, one-hot expands the categoricals)')
print('Missing values left after the pipeline:', int(Xt.isna().sum().sum()))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(tr['pm25_lag_0'], bins=80, ax=axes[0], color='gray')
axes[0].set(title='pm25_lag_0 – before preprocessing (Hanoi, train)', xlabel='µg/m³')
sns.histplot(Xt['pm__pm25_lag_0'], bins=80, ax=axes[1], color='teal')
axes[1].set(title='After IQRClipper + StandardScaler', xlabel='z-score')
plt.tight_layout()
plt.show()

## 3. ANALYSIS AND MODELING

### 3.1. Models compared
| Model | Role | Why |
|---|---|---|
| **Persistence (baseline)** | `ŷ(t+24) = pm25(t)` | Standard time series baseline: an ML model is only useful if it **beats** it |
| **Ridge Regression** | Regularized linear model | Simple and interpretable; tests how linear the relationship is |
| **Random Forest** | Tree ensemble (bagging) | Captures non-linear effects and weather × hour interactions |
| **HistGradientBoosting** | Tree ensemble (boosting) | Strong and fast on large tabular data |

**Hyperparameter tuning:** `GridSearchCV` with `TimeSeriesSplit(n_splits=3)` on the training set (the validation fold always comes
**after** the training fold), scored by `MAE`. Random Forest uses a fixed configuration (limited `max_depth` and `min_samples_leaf`
against overfitting) to keep the run time reasonable on Colab.

The `CityForecaster` class wraps the whole workflow for **one** city so that it can be repeated for both.

In [ ]:
class PersistenceBaseline(BaseEstimator, RegressorMixin):
    """Forecast PM2.5 HORIZON hours ahead with the current value."""

    def fit(self, X, y=None):
        return self

    def predict(self, X):
        return np.asarray(X['pm25_lag_0'], dtype=float)


def regression_metrics(y_true, y_pred):
    return {'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'R2': r2_score(y_true, y_pred)}


def make_models():
    return {
        'Persistence (baseline)': PersistenceBaseline(),
        'Ridge': Pipeline([('pre', build_preprocessor()), ('model', Ridge())]),
        'RandomForest': Pipeline([('pre', build_preprocessor()), ('model', RandomForestRegressor(
            n_estimators=200, max_depth=12, min_samples_leaf=10, max_features=0.5,
            n_jobs=-1, random_state=RANDOM_STATE))]),
        'HistGradientBoosting': Pipeline([('pre', build_preprocessor()),
                                          ('model', HistGradientBoostingRegressor(random_state=RANDOM_STATE))]),
    }


PARAM_GRIDS = {
    'Ridge': {'model__alpha': [0.1, 1, 10, 100]},
    'HistGradientBoosting': {'model__learning_rate': [0.03, 0.1],
                             'model__max_iter': [200, 400],
                             'model__max_leaf_nodes': [15, 31],
                             'model__l2_regularization': [0.0, 1.0]},
}


class CityForecaster:
    def __init__(self, city, train, val, test):
        self.city, self.train, self.val, self.test = city, train, val, test
        self.models = make_models()
        self.best_params = {}

    def tune(self):
        for name, grid in PARAM_GRIDS.items():
            search = GridSearchCV(self.models[name], grid, cv=TimeSeriesSplit(n_splits=3),
                                  scoring='neg_mean_absolute_error', n_jobs=-1)
            search.fit(self.train[FEATURES], self.train[TARGET])
            self.models[name] = search.best_estimator_
            self.best_params[name] = search.best_params_
        return self

    def evaluate_on_validation(self):
        rows = []
        for name, m in self.models.items():
            m.fit(self.train[FEATURES], self.train[TARGET])
            rows.append({'city': self.city, 'model': name,
                         **regression_metrics(self.val[TARGET], m.predict(self.val[FEATURES]))})
        return pd.DataFrame(rows)

    def evaluate_on_test(self):
        """Retrain on Train + Validation, then evaluate once on Test."""
        full = pd.concat([self.train, self.val])
        rows, self.test_predictions = [], pd.DataFrame({'actual': self.test[TARGET]})
        for name, m in self.models.items():
            m.fit(full[FEATURES], full[TARGET])
            pred = m.predict(self.test[FEATURES])
            self.test_predictions[name] = pred
            rows.append({'city': self.city, 'model': name, **regression_metrics(self.test[TARGET], pred)})
        return pd.DataFrame(rows)


forecasters = {city: CityForecaster(city, *parts).tune() for city, parts in splits.items()}
for city, f in forecasters.items():
    print(f'{city} – best hyperparameters:')
    for name, params in f.best_params.items():
        print(f'   {name}: {params}')

### 3.2. Comparison on the Validation set

In [ ]:
val_results = pd.concat([f.evaluate_on_validation() for f in forecasters.values()], ignore_index=True)
val_table = val_results.pivot(index='model', columns='city', values=['MAE', 'RMSE', 'R2'])
display(val_table.reindex(columns=list(CITY_FILES), level=1).reindex(list(make_models())).round(3))

**Validation findings:**
- In both cities every machine learning model has a lower MAE than the Persistence baseline. The gain is large in Hanoi
  (14.97 → 12.45 with RandomForest, ≈17% lower) and small in HCMC (7.71 → 7.07 with Ridge, ≈8% lower).
- The baseline has a negative R² (-0.06 / -0.15): copying today's value as tomorrow's forecast is worse than predicting the mean.
- In Hanoi the three ML models are very close (MAE 12.45–12.61). In HCMC Ridge is best: with a weak signal (R² only 0.07–0.14),
  a strongly regularized linear model (alpha = 100) overfits less than the tree models.
- GridSearch picks a small learning rate (0.03) and few leaves (15) for HGB, i.e. it prefers simpler models. This is consistent with
  noisy data that covers only 22 months.

## 4. SUMMARY AND EXTENSIONS

### 4.1. Results on the independent Test set
Each model is retrained on Train + Validation (with its hyperparameters fixed) and evaluated **once** on Test.
The **best model of each city is chosen on the Validation set** (lowest Validation MAE, Section 3.2), not on Test,
so the Test set is used only for reporting and the error analysis below is not biased by model selection.

In [ ]:
test_results = pd.concat([f.evaluate_on_test() for f in forecasters.values()], ignore_index=True)
base_mae = test_results[test_results.model == 'Persistence (baseline)'].set_index('city')['MAE']
test_results['MAE reduction vs baseline'] = test_results.apply(lambda r: 1 - r['MAE'] / base_mae[r['city']], axis=1)
display(test_results.set_index(['city', 'model']).style.format(
    {'MAE': '{:.2f}', 'RMSE': '{:.2f}', 'R2': '{:.3f}', 'MAE reduction vs baseline': '{:+.1%}'}))
print('Test:', {c: f.test.index.min().strftime('%Y-%m-%d') + ' → ' + f.test.index.max().strftime('%Y-%m-%d')
                for c, f in forecasters.items()})
ml_val = val_results[val_results.model != 'Persistence (baseline)']
best_model = ml_val.loc[ml_val.groupby('city')['MAE'].idxmin()].set_index('city')['model']
print('Best model (chosen on Validation):', best_model.to_dict())

### 4.2. Question: *How much better than the baseline is machine learning in each city?*
**Figure 6 – Test MAE** of each model, per city (bar labels: % MAE reduction vs. the Persistence baseline).

In [ ]:
MODEL_ORDER = list(make_models())
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=test_results, x='model', y='MAE', hue='city', order=MODEL_ORDER,
            hue_order=list(CITY_FILES), palette=CITY_COLORS, ax=ax)
for container, city in zip(ax.containers, CITY_FILES):
    sub = test_results[test_results.city == city].set_index('model')
    ax.bar_label(container, labels=[f"{sub.loc[m, 'MAE reduction vs baseline']:+.0%}" for m in MODEL_ORDER],
                 fontsize=9, padding=2)
ax.set(title='Test MAE (lower is better)', xlabel='', ylabel='MAE (µg/m³)')
ax.set_ylim(0, test_results['MAE'].max() * 1.25)
ax.legend(title='', loc='upper center', ncol=2)
plt.show()

**Figure 6 caption:** On the independent Test set, the best model of each city (chosen on Validation) reduces MAE relative to the Persistence baseline by
**13.1% in Hanoi** (RandomForest: 8.32 → 7.23 µg/m³) and by **10.6% in HCMC** (Ridge: 9.53 → 8.52 µg/m³).
In Hanoi, HistGradientBoosting is marginally lower on Test (7.20), but the 0.03 µg/m³ gap is within noise and choosing it would mean selecting on Test;
the honest conclusion is that the two tree ensembles perform the same (≈13% gain).
The best model type differs between the cities: Hanoi benefits from non-linear tree ensembles, which capture interactions
between season, rain and recent dust levels, while in HCMC the linear Ridge model is the most stable (the tree models gain only 7.6–8.7%).
So **there is no single best model for both cities**, which is why we train each city separately instead of merging the data.

Note: the two Test periods differ (Hanoi 19/03–29/09/2026, HCMC 29/06–29/09/2026, because Hanoi has more missing data)
and both fall in the cleaner season, so absolute MAE values should not be compared between cities — compare the relative improvement over the baseline instead.
Hanoi's Test MAE is lower than its Validation MAE (7.2 vs 12.5) because the Test period has no winter; HCMC is the opposite (8.5 vs 7.1) because dust rises again in September.
(Tree-model results can shift by a few hundredths of µg/m³ between scikit-learn versions; Ridge and the baseline do not change.)

### 4.3. Forecast vs. actual
**Figure 7 – First 14 days of the Test set:** actual values, the baseline and the best model of each city (chosen on Validation)
(the time axis is the **forecast** time, i.e. $t + 24h$).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
for ax, (city, f) in zip(axes, forecasters.items()):
    p = f.test_predictions
    w = p.loc[: p.index.min() + pd.Timedelta(days=14)]
    target_time = w.index + pd.Timedelta(hours=HORIZON)
    ax.plot(target_time, w['actual'], color='k', lw=1.3, label='Actual')
    ax.plot(target_time, w['Persistence (baseline)'], color='gray', ls=':', label='Persistence (baseline)')
    ax.plot(target_time, w[best_model[city]], color=CITY_COLORS[city], label=best_model[city])
    ax.set(title=f'{city} – 24h-ahead forecast', ylabel='PM2.5 (µg/m³)')
    ax.legend()
plt.tight_layout()
plt.show()

### 4.4. Error analysis
A regression problem has no direct confusion matrix, so we (1) analyze the error by **hour of day** and **month**, and
(2) convert PM2.5 into **air quality levels** (US EPA 2024 breakpoints) to build a **level confusion matrix** —
showing which level the model tends to confuse with which.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for city, f in forecasters.items():
    p = f.test_predictions
    err = (p[best_model[city]] - p['actual']).abs()
    t = p.index + pd.Timedelta(hours=HORIZON)
    err.groupby(t.hour).mean().plot(ax=axes[0], marker='o', color=CITY_COLORS[city], label=city)
    by_hour, by_month = err.groupby(t.hour).mean(), err.groupby(t.strftime('%Y-%m')).mean()
    by_month.plot(ax=axes[1], marker='o', color=CITY_COLORS[city], label=city)
    print(f'{city}: MAE by hour min {by_hour.min():.1f} (h{by_hour.idxmin()}) – max {by_hour.max():.1f} '
          f'(h{by_hour.idxmax()}) | by month: {by_month.round(1).to_dict()}')
axes[0].set(title='MAE by hour of day (best model)', xlabel='Hour', ylabel='MAE (µg/m³)')
axes[1].set(title='MAE by month in the Test set (best model)', xlabel='Month', ylabel='MAE (µg/m³)')
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
AQI_BINS = [-np.inf, 9.0, 35.4, 55.4, 125.4, np.inf]
AQI_LABELS = ['Good', 'Moderate', 'USG', 'Unhealthy', 'Very unhealthy+']


def to_aqi_level(values):
    return pd.cut(pd.Series(np.asarray(values, dtype=float)), bins=AQI_BINS, labels=AQI_LABELS)


fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (city, f) in zip(axes, forecasters.items()):
    p = f.test_predictions
    actual_lvl, pred_lvl = to_aqi_level(p['actual']), to_aqi_level(p[best_model[city]])
    cm = pd.crosstab(actual_lvl, pred_lvl, rownames=['Actual'], colnames=['Predicted'], dropna=False)
    cm = cm.reindex(index=AQI_LABELS, columns=AQI_LABELS, fill_value=0)
    cm_norm = cm.div(cm.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    sns.heatmap(cm_norm, annot=cm.values, fmt='d', cmap='Blues', cbar=False, ax=ax)
    acc = (actual_lvl == pred_lvl).mean()
    ax.set_title(f'{city} – {best_model[city]} | correct level: {acc:.1%}')
    print(f'{city}: correct AQI level {acc:.1%} | within ±1 level: '
          f'{(abs(actual_lvl.cat.codes - pred_lvl.cat.codes) <= 1).mean():.1%}')
    print(cm, end='\n\n')
plt.tight_layout()
plt.show()

*USG = Unhealthy for Sensitive Groups (35.5–55.4 µg/m³).*


**Error analysis findings:**
- **By hour:** Hanoi's error is fairly even across the day (6.1–8.6 µg/m³). HCMC's error is highest during the daytime (peak 9.9 at 14h),
  when traffic and convection make PM2.5 fluctuate most.
- **By month:** the error grows towards the end of the Test period (HCMC: 3.2 in late June → 10.9 in September 2026), as pollution starts rising again.
  The model learned mostly from the earlier distribution, so it reacts slowly to seasonal change.
- **By AQI level:** the correct level is predicted for 82.2% (Hanoi) and 76.4% (HCMC) of hours; 99.1% / 98.6% are within ±1 level.
  The main error is **under-forecasting polluted hours**: in HCMC, 281 of 359 "USG" hours are predicted as "Moderate", and 29 of 71 "Unhealthy" hours
  are also predicted as "Moderate". The model never predicts "Good", i.e. it pulls forecasts towards the mean.
- **Practical consequence:** the model is useful for following the general trend, but **not yet reliable as an early-warning tool for severe pollution episodes** —
  a case where a missed alarm costs more than a false alarm.

### 4.5. Feature importance (permutation importance)
Measures how much the Test MAE increases when each original feature of the best model is randomly shuffled.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (city, f) in zip(axes, forecasters.items()):
    model = f.models[best_model[city]]
    imp = permutation_importance(model, f.test[FEATURES], f.test[TARGET], n_repeats=5,
                                 scoring='neg_mean_absolute_error', random_state=RANDOM_STATE)
    s = pd.Series(imp.importances_mean, index=FEATURES).nlargest(10).sort_values()
    s.plot.barh(ax=ax, color=CITY_COLORS[city])
    print(f'{city} – top 5:', s[::-1].head(5).round(2).to_dict())
    ax.set(title=f'{city} – Top 10 features ({best_model[city]})', xlabel='MAE increase (µg/m³)')
plt.tight_layout()
plt.show()

**Findings:** in both cities **the current PM2.5 level (`pm25_lag_0`) is the most important feature**, consistent with the high autocorrelation.
The cities differ in the next features:
- **Hanoi:** the recent PM2.5 history (6h mean, value 6h ago), **rain over the last 24h** (`rain_24h`) and the **season**. Hanoi's dust depends on the seasonal regime and on rain washing it out.
- **HCMC:** **temperature**, the **hour of day** (`hour_cos`) and **wind speed**. HCMC's dust follows the daily activity/traffic rhythm and how well the wind disperses it.

This matches the EDA (Hanoi has a strong seasonal pattern; HCMC has a rush-hour peak and a strong correlation with wind)
and is further evidence that each city should have its own model.

### 4.6. Inference demo: `predict_next`
Takes the hourly observation history up to time $t$ and returns the PM2.5 forecast at $t + 24h$ together with its air quality level.
Demo: we use a history ending 24 hours before the last available observation, then compare with the known actual value.

In [ ]:
def predict_next(city, history):
    """history: hourly DataFrame (same columns as the cleaned data), ending at time t."""
    model = forecasters[city].models[best_model[city]]
    x_last = build_features(history, city).iloc[[-1]]
    value = float(model.predict(x_last[FEATURES])[0])
    return {'city': city, 'model': best_model[city],
            'forecast_time': x_last.index[0] + pd.Timedelta(hours=HORIZON),
            'pm25_forecast': round(value, 1), 'aqi_level': str(to_aqi_level([value])[0])}


demo_rows = []
for city, df in clean.items():
    t_end = df['pm25'].last_valid_index() - pd.Timedelta(hours=HORIZON)
    history = df.loc[t_end - pd.Timedelta(days=7): t_end]
    res = predict_next(city, history)
    actual = df.loc[res['forecast_time'], 'pm25']
    res.update({'pm25_actual': round(float(actual), 1), 'aqi_actual': str(to_aqi_level([actual])[0])})
    demo_rows.append(res)
display(pd.DataFrame(demo_rows))

### 4.7. Conclusions, limitations and future work


**Conclusions:**
- We built a 24h-ahead PM2.5 forecasting pipeline, applied identically but **trained independently** for Hanoi and HCMC, using real
  monitoring data from OpenAQ combined with Open-Meteo weather (12/2024 – 09/2026).
- On the Test set, the best model reduces MAE by **13.1% in Hanoi (RandomForest, chosen on Validation)** and **10.6% in HCMC (Ridge)**
  compared with the Persistence baseline, and predicts the correct AQI level in 76–82% of hours.
- The two cities have different pollution mechanisms: Hanoi is driven by season and rain, HCMC by the daily cycle and wind.

**Limitations:**
- Only **one station** per city. The HCMC station is a low-cost sensor (AirGradient) that may be less accurate at high humidity.
- Hanoi has a lot of missing data (32.8%, mostly from late 2025), so its Test set contains no winter — the most polluted season.
- Only 22 months of data (fewer than 2 seasonal cycles), so seasonal patterns are hard to learn.
- The models tend to under-forecast high-pollution episodes. Weather features use observations at $t$, not weather forecasts for $t+24h$.

**Future work:**
- Add **weather forecasts** for the target time (Open-Meteo forecast API) and data from more stations in each city.
- Use a loss function / sample weights that emphasize high-pollution hours, or add an "exceeds threshold or not" classifier for early warning.
- Try dedicated time series models (SARIMAX, multi-horizon LightGBM, LSTM) once longer data is available.